# Notebook 11 — Hybrid Mode Deep Dive

**What you'll learn:**
- How the hybrid feedback loop works step by step
- Training quality scoring: effectiveness, efficiency, robustness
- Adaptive loss weighting in action
- When to use each mode

**Prerequisites:** Notebook 10 (Lang-PINN intro)

**Time:** ~30 minutes

## The Hybrid Feedback Loop

Hybrid mode is the most powerful Lang-PINN mode. Here's what happens:

```
LLM generates code
  → pinn library executes it
  → Feedback Agent scores quality
  → If quality < threshold: LLM gets error/score, refines code
  → Repeat until quality >= threshold or max iterations
```

The key insight: **the LLM is the code author, the library is the runtime, the Feedback Agent is the quality gate.**

## Understanding Quality Scoring

The Feedback Agent (`pinn.feedback`) evaluates training quality across three dimensions. Let's see how each works.

In [ ]:
from pinn import evaluate_quality

# Simulate a good training run: loss drops steadily
good_history = []
for i in range(1000):
    loss = 1.0 * (0.995 ** i)  # smooth exponential decay
    good_history.append({"total": loss, "physics": loss * 0.8, "ic": loss * 0.2})

quality = evaluate_quality(good_history)
print("Good training run:")
for k, v in quality.items():
    print(f"  {k}: {v:.4f}" if isinstance(v, float) else f"  {k}: {v}")

In [ ]:
import random

# Simulate a bad training run: oscillating, not converging
random.seed(42)
bad_history = []
for _i in range(1000):
    loss = 0.5 + 0.3 * random.random()  # noisy, stuck around 0.5-0.8
    bad_history.append({"total": loss, "physics": loss * 0.6, "ic": loss * 0.4})

quality_bad = evaluate_quality(bad_history)
print("Bad training run:")
for k, v in quality_bad.items():
    print(f"  {k}: {v:.4f}" if isinstance(v, float) else f"  {k}: {v}")

print("\nQuality score comparison:")
print(f"  Good: {quality['quality_score']:.3f}")
print(f"  Bad:  {quality_bad['quality_score']:.3f}")

The quality score combines:
- **Effectiveness** (40%): how low is the final loss? Uses log-scale normalization.
- **Efficiency** (30%): how quickly did it converge? Fraction of epochs before plateau.
- **Robustness** (30%): how smooth was training? Measures loss oscillation.

In hybrid mode, the orchestrator checks `quality_score >= 0.5`. If below, it asks the LLM to refine.

## Training Health Monitor

The `TrainingHealthMonitor` tracks health metrics during training:

In [ ]:
import torch
from pinn import PINN, PINNTrainer, TrainingHealthMonitor

# Build a tiny model
model = PINN(input_dim=1, hidden_layers=2, hidden_neurons=16)
monitor = TrainingHealthMonitor(model, window=50)

# Set up a simple problem: u' + u = 0
t = torch.linspace(0, 1, 50).view(-1, 1).requires_grad_(True)

def physics_loss(model):
    u = model(t)
    u_t = torch.autograd.grad(u, t, torch.ones_like(u), create_graph=True)[0]
    return torch.mean((u_t + u) ** 2)

def ic_loss(model):
    t0 = torch.tensor([[0.0]], requires_grad=True)
    return (model(t0) - 1.0) ** 2

# Train with monitor
trainer = PINNTrainer(model)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

trainer.train(
    n_epochs=500,
    optimizer=optimizer,
    loss_functions={"ic": ic_loss, "physics": physics_loss},
    callbacks=[monitor],
    verbose=False,
)

# Check health report
report = monitor.report()
print("Health Report:")
for k, v in report.items():
    print(f"  {k}: {v:.4f}" if isinstance(v, float) else f"  {k}: {v}")

Key health metrics:
- **Loss smoothness**: `1 - Std(delta_L) / Mean(L)`. Close to 1.0 = smooth training.
- **Gradient healthy**: are gradients in a reasonable range? `eps <= ||grad|| <= kappa`
- **Convergence epoch**: first epoch where loss drops below threshold

## Adaptive Loss Weighting

When one loss term dominates, gradient starvation kills the other terms. The `AdaptiveLossWeighter` fixes this by dynamically rebalancing:

In [ ]:
from pinn import AdaptiveLossWeighter

# Start with equal weights
weights = {"ic": 1.0, "physics": 1.0}
print(f"Initial weights: {weights}")

# Create weighter with aggressive rebalancing (every 100 epochs for demo)
weighter = AdaptiveLossWeighter(weights, rebalance_every=100, ratio_threshold=3.0)

# Simulate: IC loss is 100x larger than physics loss
for epoch in range(500):
    fake_losses = {"ic": 10.0, "physics": 0.1}
    weighter(epoch, fake_losses)

print(f"After rebalancing: {weights}")
print("\nThe IC weight decreased and physics weight increased")
print("to compensate for the imbalance — preventing gradient starvation.")

## Putting It Together: The SolveResult

When the orchestrator runs, it produces a `SolveResult` with everything:

In [ ]:
from lang_pinn import Orchestrator, PDESpec

# Build a spec for a simple ODE
decay_spec = PDESpec(
    name="Exponential Decay",
    equation="u_t + u = 0",
    independent_vars=["t"],
    dependent_var="u",
    order=1,
    spatial_dim=0,
    domain={"t": (0.0, 3.0)},
    initial_conditions=["u(0) = 1"],
)

# Library mode (no LLM needed)
orch = Orchestrator(mode="library")
result = orch.solve_from_spec(decay_spec)

print(f"Spec: {result.spec.name}")
print(f"Architecture: {result.architecture.hidden_layers}x{result.architecture.hidden_neurons}")
print(f"Mode: {result.mode}")
print(f"Executed: {result.executed}")
print(f"Quality: {result.quality_score}")
print(f"Iterations: {result.iterations}")
print("\nCode preview (first 5 lines):")
for line in result.code.split("\n")[:5]:
    print(f"  {line}")

## When to Use Each Mode

| Situation | Recommended Mode | Why |
|-----------|-----------------|-----|
| Production deployment | **library** | Deterministic, reproducible, no LLM dependency |
| Exploring a new PDE | **hybrid** | LLM flexibility + library quality gate |
| Quick prototyping | **code-agent** | Maximum flexibility, accept imperfections |
| Teaching / demos | **library** | Predictable output for explanations |
| Unusual PDE types | **hybrid** | LLM can adapt beyond our 11 experiment templates |

The golden rule: **start with library mode** to see the baseline. If it doesn't handle your problem well, try hybrid.

## Key Takeaways

1. **Quality scoring** gives a 0-1 score combining effectiveness, efficiency, and robustness
2. **Health monitoring** detects gradient issues, convergence stalls, and loss oscillation during training
3. **Adaptive weights** prevent gradient starvation by rebalancing when one loss dominates
4. **Hybrid mode** chains these together: generate → execute → score → refine
5. **The quality threshold** (0.5 by default) determines when to accept vs refine

**Next:** Notebook 12 — bring your own PDE and solve it end-to-end.